# Импорты

In [ ]:
!pip install catboost
!pip install pymorphy3
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 31.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import re
from scipy.sparse import hstack, csr_matrix
import optuna

import pymorphy3
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)


In [3]:
from nltk.corpus import stopwords
import nltk

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Загрузка и исследование данных

In [4]:
df = pd.read_excel('nerd_analytics_v25.xlsx', sheet_name='reviews')

In [5]:
df = df[['id', 'comment', 'sentiment', 'rating', 'keywords_positive', 'keywords_neutral', 'keywords_negative', 'product', 'final_category']]

In [6]:
df.shape[0]

1235

In [7]:
df[['id', 'comment', 'sentiment', 'keywords_positive', 'keywords_neutral', 'keywords_negative', 'product', 'final_category']].describe()

,id,comment,sentiment,keywords_positive,keywords_neutral,keywords_negative,product,final_category
count,1235,1235,1235,815,397,189,1235,1235
unique,1235,65,3,29,12,16,6,5
top,266a542e-a9f3-4127-916e-76e017b10e1b,"Ответили быстро, не пришлось долго ждать",positive,приятно,целом,долго,аналитический модуль,качество ответа
freq,1,45,770,59,66,69,249,286


In [8]:
df.describe()

,rating
count,1235.000000
mean,3.710121
std,1.155714
min,1.000000
25%,3.000000
50%,4.000000
75%,5.000000
max,5.000000


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1235 entries, 0 to 1234
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 1235 non-null   object
 1   comment            1235 non-null   object
 2   sentiment          1235 non-null   object
 3   rating             1235 non-null   int64 
 4   keywords_positive  815 non-null    object
 5   keywords_neutral   397 non-null    object
 6   keywords_negative  189 non-null    object
 7   product            1235 non-null   object
 8   final_category     1235 non-null   object
dtypes: int64(1), object(8)
memory usage: 87.0+ KB


In [10]:
df['product'].value_counts()

,count
product,
аналитический модуль,249
платёжный сервис,210
API интеграция,206
мобильное приложение,204
веб-сервис,193
личный кабинет,173


In [11]:
df.isna().sum(), df.duplicated().sum()

(id                      0
 comment                 0
 sentiment               0
 rating                  0
 keywords_positive     420
 keywords_neutral      838
 keywords_negative    1046
 product                 0
 final_category          0
 dtype: int64,
 np.int64(0))

# Определение общей тональности отзыва

## Подготовка данных

In [12]:
df['final_category'].value_counts()

,count
final_category,
качество ответа,286
скорость решения,275
общее впечатление,271
вежливость,207
техническая компетентность,196


In [13]:
assert set([len(i) for i in df.groupby('comment')['final_category'].unique().reset_index()['final_category']]) == {1}

In [14]:
df.groupby(['product', 'final_category'])['id'].count()

product               final_category            
API интеграция        вежливость                    41
                      качество ответа               43
                      общее впечатление             42
                      скорость решения              46
                      техническая компетентность    34
аналитический модуль  вежливость                    36
                      качество ответа               61
                      общее впечатление             54
                      скорость решения              59
                      техническая компетентность    39
веб-сервис            вежливость                    27
                      качество ответа               59
                      общее впечатление             38
                      скорость решения              43
                      техническая компетентность    26
личный кабинет        вежливость                    37
                      качество ответа               29
                      общее впечатление             41
                      скорость решения              35
                      техническая компетентность    31
мобильное приложение  вежливость                    36
                      качество ответа               51
                      общее впечатление             45
                      скорость решения              39
                      техническая компетентность    33
платёжный сервис      вежливость                    30
                      качество ответа               43
                      общее впечатление             51
                      скорость решения              53
                      техническая компетентность    33
Name: id, dtype: int64

In [15]:
df1 = df.copy()

In [16]:
morph = pymorphy3.MorphAnalyzer()
russian_stopwords = stopwords.words('russian')
bad_stopwords = ['не', 'нет', 'ни', 'без']

russian_stopwords = [
    word for word in russian_stopwords
    if word not in bad_stopwords
]

def clean_text(text):

    text = text.lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^а-яa-zё\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    lemmas = []

    for word in words:
        if word not in russian_stopwords:
            lemma = morph.parse(word)[0].normal_form
            lemmas.append(lemma)

    return " ".join(lemmas)

df1['clean_comment'] = df1['comment'].apply(clean_text)

In [17]:
RANDOM_STATE = 42

## Подбираем модель и гиперпараметры для решения задачи через Optuna


In [18]:
cat_df = df1[['clean_comment', 'sentiment', 'rating', 'final_category']].copy()
X_all = cat_df[['clean_comment', 'sentiment', 'rating']]
y_all = cat_df['final_category']
groups_all = cat_df['clean_comment']

N_SPLITS = 5
CV_RANDOM_STATES = [RANDOM_STATE, 101, 202, 303, 404]


def build_features(train_part, valid_part):
    word_vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2), min_df=2, max_df=0.95)
    char_vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=20000)
    rating_scaler = StandardScaler()
    try:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown='ignore', sparse=True)

    X_train_word = word_vectorizer.fit_transform(train_part['clean_comment'])
    X_valid_word = word_vectorizer.transform(valid_part['clean_comment'])
    X_train_char = char_vectorizer.fit_transform(train_part['clean_comment'])
    X_valid_char = char_vectorizer.transform(valid_part['clean_comment'])
    X_train_sentiment = encoder.fit_transform(train_part[['sentiment']])
    X_valid_sentiment = encoder.transform(valid_part[['sentiment']])
    X_train_rating = csr_matrix(rating_scaler.fit_transform(train_part[['rating']]))
    X_valid_rating = csr_matrix(rating_scaler.transform(valid_part[['rating']]))

    return hstack([X_train_word, X_train_char, X_train_sentiment, X_train_rating]), hstack([X_valid_word, X_valid_char, X_valid_sentiment, X_valid_rating])


def build_model_and_params(trial):
    model_type = trial.suggest_categorical('model_type', ['SVC', 'LogReg', 'CatBoost'])

    if model_type == 'SVC':
        params = {
            'C': trial.suggest_float('svc_C', 0.1, 5.0, log=True),
            'kernel': trial.suggest_categorical('svc_kernel', ['linear', 'rbf']),
            'gamma': 'scale',
            'class_weight': trial.suggest_categorical('svc_class_weight', [None, 'balanced']),
        }
        return model_type, params

    if model_type == 'LogReg':
        params = {
            'C': trial.suggest_float('lr_C', 0.1, 10.0, log=True),
            'solver': trial.suggest_categorical('lr_solver', ['lbfgs', 'saga']),
            'class_weight': trial.suggest_categorical('lr_class_weight', [None, 'balanced']),
            'max_iter': 3000,
            'n_jobs': -1,
            'random_state': RANDOM_STATE,
        }
        return model_type, params

    params = {
        'iterations': trial.suggest_int('cb_iterations', 200, 900),
        'depth': trial.suggest_int('cb_depth', 4, 10),
        'learning_rate': trial.suggest_float('cb_learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_int('cb_l2_leaf_reg', 1, 12),
        'random_strength': trial.suggest_float('cb_random_strength', 1.0, 10.0),
        'bagging_temperature': trial.suggest_float('cb_bagging_temperature', 0.0, 10.0),
        'border_count': trial.suggest_int('cb_border_count', 64, 255),
        'auto_class_weights': trial.suggest_categorical('cb_auto_class_weights', ['Balanced', None]),
    }
    return model_type, params


def make_model(model_type, model_params):
    if model_type == 'SVC':
        return SVC(**model_params)

    if model_type == 'LogReg':
        return LogisticRegression(**model_params)

    return CatBoostClassifier(
        **model_params,
        loss_function='MultiClass',
        eval_metric='TotalF1',
        random_state=RANDOM_STATE,
        task_type='GPU',
        verbose=False,
    )


def get_cv_scores(model_type, model_params, data, cv_random_states):
    fold_scores = []

    for cv_seed in cv_random_states:
        cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_seed)

        for tr_fold_idx, val_fold_idx in cv.split(
            data[['clean_comment', 'sentiment', 'rating']],
            data['final_category'],
            groups=data['clean_comment'],
        ):
            tr_part = data.iloc[tr_fold_idx]
            val_part = data.iloc[val_fold_idx]

            X_tr, X_val = build_features(tr_part, val_part)
            y_tr = tr_part['final_category']
            y_val = val_part['final_category']

            model = make_model(model_type, model_params)
            model.fit(X_tr, y_tr, verbose=False) if model_type == 'CatBoost' else model.fit(X_tr, y_tr)
            pred = model.predict(X_val).flatten() if model_type == 'CatBoost' else model.predict(X_val)

            fold_scores.append(f1_score(y_val, pred, average='macro'))

    return fold_scores


def objective(trial):
    model_type, model_params = build_model_and_params(trial)
    fold_scores = get_cv_scores(model_type, model_params, cat_df, CV_RANDOM_STATES)
    return float(np.mean(fold_scores))


sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(objective, n_trials=40)

print('BEST MEAN CV SCORE:')
print(study.best_value)
print('BEST PARAMS:')
print(study.best_params)

[I 2026-05-25 15:45:08,803] A new study created in memory with name: no-name-22d4b42c-76f2-4ba9-96d5-d8d948075c0b
[I 2026-05-25 15:45:26,372] Trial 0 finished with value: 0.39280797048792465 and parameters: {'model_type': 'LogReg', 'lr_C': 1.5751320499779735, 'lr_solver': 'lbfgs', 'lr_class_weight': 'balanced'}. Best is trial 0 with value: 0.39280797048792465.
[I 2026-05-25 15:45:36,373] Trial 1 finished with value: 0.39021279803858805 and parameters: {'model_type': 'LogReg', 'lr_C': 8.706020878304859, 'lr_solver': 'lbfgs', 'lr_class_weight': 'balanced'}. Best is trial 0 with value: 0.39280797048792465.
[I 2026-05-25 15:45:40,529] Trial 2 finished with value: 0.3815365537168651 and parameters: {'model_type': 'LogReg', 'lr_C': 0.38234752246751863, 'lr_solver': 'lbfgs', 'lr_class_weight': 'balanced'}. Best is trial 0 with value: 0.39280797048792465.
[I 2026-05-25 15:45:45,945] Trial 3 finished with value: 0.3797039152868809 and parameters: {'model_type': 'LogReg', 'lr_C': 1.0677482709481

BEST MEAN CV SCORE:
0.4043711798725888
BEST PARAMS:
{'model_type': 'LogReg', 'lr_C': 9.839975622808534, 'lr_solver': 'saga', 'lr_class_weight': 'balanced'}


## Обучаем лучшую модель

In [19]:
best_params = dict(study.best_params)
best_model_type = best_params.pop('model_type')

if best_model_type == 'SVC':
    model_params = {
        'C': best_params['svc_C'],
        'kernel': best_params['svc_kernel'],
        'gamma': 'scale',
        'class_weight': best_params['svc_class_weight'],
    }
elif best_model_type == 'LogReg':
    model_params = {
        'C': best_params['lr_C'],
        'solver': best_params['lr_solver'],
        'class_weight': best_params['lr_class_weight'],
        'max_iter': 3000,
        'n_jobs': -1,
        'random_state': RANDOM_STATE,
    }
else:
    model_params = {
        'iterations': best_params['cb_iterations'],
        'depth': best_params['cb_depth'],
        'learning_rate': best_params['cb_learning_rate'],
        'l2_leaf_reg': best_params['cb_l2_leaf_reg'],
        'random_strength': best_params['cb_random_strength'],
        'bagging_temperature': best_params['cb_bagging_temperature'],
        'border_count': best_params['cb_border_count'],
        'auto_class_weights': best_params['cb_auto_class_weights'],
    }

cv_results = []
y_true_all = []
y_pred_all = []

for cv_seed in CV_RANDOM_STATES:
    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=cv_seed)

    for fold, (tr_fold_idx, val_fold_idx) in enumerate(
        cv.split(X_all, y_all, groups=groups_all),
        start=1,
    ):
        tr_part = cat_df.iloc[tr_fold_idx]
        val_part = cat_df.iloc[val_fold_idx]

        X_tr, X_val = build_features(tr_part, val_part)
        y_tr = tr_part['final_category']
        y_val = val_part['final_category']

        model = make_model(best_model_type, model_params)
        model.fit(X_tr, y_tr, verbose=False) if best_model_type == 'CatBoost' else model.fit(X_tr, y_tr)
        pred = model.predict(X_val).flatten() if best_model_type == 'CatBoost' else model.predict(X_val)

        cv_results.append({
            'seed': cv_seed,
            'fold': fold,
            'accuracy': accuracy_score(y_val, pred),
            'macro_f1': f1_score(y_val, pred, average='macro'),
        })
        y_true_all.extend(y_val)
        y_pred_all.extend(pred)

cv_results_df = pd.DataFrame(cv_results)
y_true_cv = pd.Series(y_true_all, name='true')
y_pred_cv = pd.Series(y_pred_all, name='pred')

X_full, _ = build_features(cat_df, cat_df)
best_model = make_model(best_model_type, model_params)
best_model.fit(X_full, y_all, verbose=False) if best_model_type == 'CatBoost' else best_model.fit(X_full, y_all)

LogisticRegression(C=9.839975622808534, class_weight='balanced', max_iter=3000,
                   n_jobs=-1, random_state=42, solver='saga')

## Результат на тестовой выборке

In [20]:
accuracy_mean = cv_results_df['accuracy'].mean()
accuracy_std = cv_results_df['accuracy'].std()
macro_f1_mean = cv_results_df['macro_f1'].mean()
macro_f1_std = cv_results_df['macro_f1'].std()

print(f'\nBest model type: {best_model_type}')
print(f'CV seeds: {CV_RANDOM_STATES}')
print(f'Accuracy: {accuracy_mean:.4f} +/- {accuracy_std:.4f}')
print(f'Macro F1: {macro_f1_mean:.4f} +/- {macro_f1_std:.4f}')
print('\nFOLD METRICS SUMMARY:\n')
print(cv_results_df[['accuracy', 'macro_f1']].describe())
print('\nCLASSIFICATION REPORT OVER ALL CV PREDICTIONS:\n')
print(classification_report(y_true_cv, y_pred_cv, zero_division=0))
print('\nCONFUSION MATRIX OVER ALL CV PREDICTIONS:\n')
print(confusion_matrix(y_true_cv, y_pred_cv))


Best model type: LogReg
CV seeds: [42, 101, 202, 303, 404]
Accuracy: 0.4568 +/- 0.1744
Macro F1: 0.4044 +/- 0.1804

FOLD METRICS SUMMARY:

        accuracy   macro_f1
count  25.000000  25.000000
mean    0.456832   0.404371
std     0.174412   0.180426
min     0.151316   0.089157
25%     0.305842   0.277710
50%     0.475460   0.411359
75%     0.584416   0.526743
max     0.771930   0.771229

CLASSIFICATION REPORT OVER ALL CV PREDICTIONS:

                            precision    recall  f1-score   support

                вежливость       0.61      0.59      0.60      1035
           качество ответа       0.41      0.47      0.44      1430
         общее впечатление       0.46      0.50      0.48      1355
          скорость решения       0.38      0.40      0.39      1375
техническая компетентность       0.59      0.39      0.47       980

                  accuracy                           0.47      6175
                 macro avg       0.49      0.47      0.48      6175
             

## Сохранение

In [21]:
import joblib

In [22]:
joblib.dump(best_model, 'reviews_categories.pkl')

['reviews_categories.pkl']